In [1]:
import os, numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
FILE="Water Quality.csv"
def locate(name):
    if os.path.isfile(name): return os.path.abspath(name)
    for base in [os.getcwd(),os.path.expanduser("~/Desktop"),os.path.expanduser("~/Downloads"),os.path.expanduser("~/Documents")]:
        if os.path.isdir(base):
            for root,dirs,files in os.walk(base):
                dirs[:]=[d for d in dirs if d not in {".git","node_modules",".venv","venv","__pycache__"}]
                for f in files:
                    if f.lower()==name.lower(): return os.path.join(root,f)
    from tkinter import Tk,filedialog
    r=Tk(); r.withdraw(); r.attributes("-topmost",True); p=filedialog.askopenfilename(title="Select Water Quality CSV",filetypes=[("CSV files","*.csv")]); r.destroy()
    return p or None
path=locate(FILE)
if not path: raise FileNotFoundError("Select Water Quality.csv or update FILE.")
df=pd.read_csv(path,low_memory=False); df.columns=df.columns.str.strip()
req={"MonitoringLocationID","ActivityStartDate","CharacteristicName","ResultValue"}
if not req.issubset(df.columns): raise ValueError(f"Missing columns: {sorted(req-set(df.columns))}")
df["ActivityStartDate"]=pd.to_datetime(df["ActivityStartDate"],errors="coerce"); df["ResultValue"]=pd.to_numeric(df["ResultValue"],errors="coerce")
df=df.dropna(subset=["MonitoringLocationID","ActivityStartDate","CharacteristicName","ResultValue"]).copy()
df["YearMonth"]=df["ActivityStartDate"].dt.to_period("M").astype(str)
m=df.groupby(["MonitoringLocationID","YearMonth","CharacteristicName"])["ResultValue"].median().reset_index()
wide=m.pivot_table(index=["MonitoringLocationID","YearMonth"],columns="CharacteristicName",values="ResultValue").reset_index(); wide.columns.name=None
ids=["MonitoringLocationID","YearMonth"]; num=wide.drop(columns=ids).apply(pd.to_numeric,errors="coerce")
num=num.loc[:,[c for c in num if num[c].notna().any() and num[c].nunique(dropna=True)>1]]
if num.shape[1]<2 or len(num)<3: raise ValueError("Need at least 3 records and 2 varying parameters.")
imp=SimpleImputer(strategy="median").fit_transform(num)
# Entropy weights after min-max scaling: statistical dispersion, not regulatory importance.
z=np.zeros_like(imp,dtype=float)
for j in range(imp.shape[1]):
    lo,hi=imp[:,j].min(),imp[:,j].max(); z[:,j]=(imp[:,j]-lo)/(hi-lo) if hi>lo else 0
P=z+1e-12; P=P/P.sum(axis=0,keepdims=True); ent=-(P*np.log(P)).sum(axis=0)/np.log(len(P))
div=1-ent; ew=div/div.sum() if div.sum()>0 else np.repeat(1/len(div),len(div))
sc=StandardScaler().fit_transform(imp); pca=PCA().fit(sc)
pc=(np.abs(pca.components_)*pca.explained_variance_ratio_[:,None]).sum(axis=0); pw=pc/pc.sum() if pc.sum()>0 else np.repeat(1/len(pc),len(pc))
out=os.path.join(os.path.dirname(path),"task2_weighted_wqi_outputs"); os.makedirs(out,exist_ok=True)
wide.to_csv(os.path.join(out,"monthly_parameter_table.csv"),index=False)
pd.DataFrame({"Parameter":num.columns,"EntropyWeight":ew,"PCAWeight":pw}).sort_values("EntropyWeight",ascending=False).to_csv(os.path.join(out,"entropy_pca_weight_comparison.csv"),index=False)
pd.DataFrame({"Component":np.arange(1,len(pca.explained_variance_ratio_)+1),"ExplainedVarianceRatio":pca.explained_variance_ratio_,"CumulativeExplainedVariance":np.cumsum(pca.explained_variance_ratio_)}).to_csv(os.path.join(out,"pca_explained_variance.csv"),index=False)
# Optional approved specification CSV alongside dataset: Parameter,Weight,IdealValue,StandardValue,Direction
# Direction: lower_better or higher_better. This generic score is not NSF-WQI/regulatory certification.
specpath=os.path.join(os.path.dirname(path),"wqi_specification.csv")
if os.path.exists(specpath):
    s=pd.read_csv(specpath); need={"Parameter","Weight","IdealValue","StandardValue","Direction"}
    if not need.issubset(s.columns): raise ValueError("Specification needs columns: "+", ".join(sorted(need)))
    for c in ["Weight","IdealValue","StandardValue"]: s[c]=pd.to_numeric(s[c],errors="coerce")
    s=s.dropna(subset=list(need)); s=s[s["Parameter"].isin(num.columns)].copy()
    if s.empty: raise ValueError("No specification parameters match exact CharacteristicName values.")
    if (s.Weight<=0).any() or (s.StandardValue==s.IdealValue).any(): raise ValueError("Weights must be positive and StandardValue must differ from IdealValue.")
    if not s.Direction.isin(["higher_better","lower_better"]).all(): raise ValueError("Direction must be higher_better or lower_better.")
    total=s.Weight.sum(); numerator=pd.Series(0.,index=wide.index); denom=pd.Series(0.,index=wide.index)
    for _,r in s.iterrows():
        v=pd.to_numeric(wide[r.Parameter],errors="coerce")
        q=100*(v-r.IdealValue)/(r.StandardValue-r.IdealValue) if r.Direction=="lower_better" else 100*(r.StandardValue-v)/(r.StandardValue-r.IdealValue)
        ok=v.notna() & np.isfinite(q); numerator.loc[ok]+=q.loc[ok]*r.Weight/total; denom.loc[ok]+=r.Weight/total
    result=wide[ids].copy(); result["WQI_configurable"]=numerator/denom.replace(0,np.nan); result["WeightCoverage"]=denom
    result.to_csv(os.path.join(out,"configurable_wqi_results.csv"),index=False)
else:
    pd.DataFrame({"Parameter":["REPLACE_WITH_EXACT_PARAMETER_NAME"],"Weight":[1.0],"IdealValue":[0.0],"StandardValue":[1.0],"Direction":["lower_better"]}).to_csv(os.path.join(out,"wqi_specification_template.csv"),index=False)
    print("No approved wqi_specification.csv; WQI not calculated. Template created.")
print("Task 2 outputs:",out)
print("Entropy/PCA weights are statistical, not regulatory/health weights. Confirm WQI method and standards with your supervisor.")


No approved wqi_specification.csv; WQI not calculated. Template created.
Task 2 outputs: c:\Users\YASH\Desktop\BrainyBeam\task3 web\task2_weighted_wqi_outputs
Entropy/PCA weights are statistical, not regulatory/health weights. Confirm WQI method and standards with your supervisor.
